In [5]:
import os
if os.name == 'nt':  # Check if windows
    os.add_dll_directory(r'C:\Program Files\SuperTuxKart 1.5')

import multiprocessing as mp
import numpy as np

import torch
import torch.optim as optim
import torch.nn.functional as F
from actor import ActorNetwork
from critic import CriticNetwork
from env_worker import SingleInstance


In [6]:
# ── Model Instantiation ────────────────────────────────────────────────────────
actor_net  = ActorNetwork(state_dim=52)
critic_net = CriticNetwork(state_dim=52)

actor_net.eval()
critic_net.eval()

# Separate learning rates: critic learns faster (needs accurate value estimates)
optimizer = optim.Adam([
    {'params': actor_net.parameters(),  'lr': 3e-4},
    {'params': critic_net.parameters(), 'lr': 1e-3},
])

# ── Generalised Advantage Estimation (GAE) ─────────────────────────────────────
def compute_gae(rewards, values, dones, gamma=0.995, lam=0.95):
    """
    Compute GAE advantages and value targets.
    Returns (advantages, returns) both as FloatTensors shape (N, 1).
    """
    advantages = []
    gae        = 0.0
    next_value = 0.0   # episode ends at buffer boundary → bootstrap = 0

    for t in reversed(range(len(rewards))):
        mask   = 0.0 if dones[t] else 1.0
        delta  = rewards[t] + gamma * next_value * mask - values[t]
        gae    = delta + gamma * lam * mask * gae
        advantages.insert(0, gae)
        next_value = values[t]

    advantages = torch.FloatTensor(advantages).unsqueeze(1)
    returns    = advantages + torch.FloatTensor(values).unsqueeze(1)
    return advantages, returns


# ── PPO Update ─────────────────────────────────────────────────────────────────
def update_ppo(buffer, epochs=6, gamma=0.995, clip_epsilon=0.2, lam=0.95,
               entropy_coeff=0.01, value_coeff=0.5, max_grad_norm=0.5):
    """
    Proximal Policy Optimisation update with GAE.

    Key hyperparameters vs. previous version:
      clip_epsilon : 0.1  → 0.2   (standard PPO, allows more policy improvement)
      gamma        : 0.99 → 0.995 (longer horizon — values finishing the race)
      epochs       : 4   → 6      (more gradient steps per rollout)
      entropy_coeff: 0.05 → 0.01  (less randomness pressure as policy improves)
      GAE lambda   : 0.95          (lower variance advantage estimates)
      Actor LR     : 1e-4 → 3e-4  (faster learning)
      Critic LR    : 1e-4 → 1e-3  (much faster value function learning)
    """
    if not buffer:
        return

    states        = torch.FloatTensor(np.array([t['state']  for t in buffer]))
    # Actions: exactly 3 dimensions (Steer, Accel, Brake) — no padding
    actions       = torch.FloatTensor([[t['action'][0], t['action'][1], t['action'][2]]
                                       for t in buffer])
    rewards_list  = [t['reward'] for t in buffer]
    values_list   = [t['value']  for t in buffer]
    dones_list    = [t['done']   for t in buffer]
    old_log_probs = torch.FloatTensor([t['log_prob'] for t in buffer]).unsqueeze(1)

    # GAE advantages and discounted returns
    advantages, returns = compute_gae(rewards_list, values_list, dones_list, gamma, lam)

    # Normalise advantages (zero-mean, unit-std)
    advantages = (advantages - advantages.mean()) / (advantages.std(unbiased=False) + 1e-8)

    actor_net.train()
    critic_net.train()

    for epoch in range(epochs):
        if torch.isnan(states).any():
            print('NaN detected in states buffer! Skipping PPO update.')
            break

        action_dists   = actor_net(states)
        new_values     = critic_net(states)
        new_log_probs  = action_dists.log_prob(actions).sum(dim=-1, keepdim=True)
        entropy        = action_dists.entropy().sum(dim=-1, keepdim=True).mean()

        ratio  = torch.exp(new_log_probs - old_log_probs)
        surr1  = ratio * advantages
        surr2  = torch.clamp(ratio, 1.0 - clip_epsilon, 1.0 + clip_epsilon) * advantages

        actor_loss  = -torch.min(surr1, surr2).mean()
        critic_loss = F.mse_loss(new_values, returns)
        loss        = actor_loss + value_coeff * critic_loss - entropy_coeff * entropy

        optimizer.zero_grad()
        loss.backward()

        # Gradient clipping
        actor_grad_norm  = torch.nn.utils.clip_grad_norm_(actor_net.parameters(),  max_grad_norm)
        critic_grad_norm = torch.nn.utils.clip_grad_norm_(critic_net.parameters(), max_grad_norm)

        optimizer.step()

    actor_net.eval()
    critic_net.eval()

    # Diagnostics on the last epoch
    with torch.no_grad():
        final_log_probs = actor_net(states).log_prob(actions).sum(dim=-1, keepdim=True)
        kl_approx = (old_log_probs - final_log_probs).mean().item()
    print(f'  [PPO] actor_loss={actor_loss.item():.4f}  '
          f'critic_loss={critic_loss.item():.4f}  '
          f'entropy={entropy.item():.4f}  '
          f'approx_kl={kl_approx:.4f}  '
          f'actor_grad={actor_grad_norm:.4f}')


In [7]:
def main():
    PATIENCE_LIMIT    = 60
    NUM_EPISODES      = 500
    NUM_WORKERS       = 5
    STEPS_PER_EPISODE = 1000

    best_reward     = float('-inf')
    patience_counter = 0

    try:
        for episode in range(NUM_EPISODES):
            buffer               = []
            total_episode_reward = 0.0
            ProcessList          = []
            ConList              = []

            # Spawn worker processes
            for i in range(NUM_WORKERS):
                ParentCon, ChildCon = mp.Pipe()
                process = mp.Process(target=SingleInstance, args=(i, ChildCon))
                ProcessList.append(process)
                ConList.append(ParentCon)
                process.start()

            # Receive initial observations from all workers
            BatchStates = []
            BatchDones  = []
            for con in ConList:
                np_obs, reward, RaceDone = con.recv()
                BatchStates.append(np_obs)
                BatchDones.append(RaceDone)

            # ── Rollout loop ───────────────────────────────────────────────────
            for step in range(STEPS_PER_EPISODE):
                state_tensor = torch.FloatTensor(np.array(BatchStates))

                if torch.isnan(state_tensor).any():
                    print('NaN in engine observations! Terminating episode.')
                    break

                with torch.no_grad():
                    action_dist    = actor_net(state_tensor)
                    sampled_action = action_dist.sample()
                    state_value    = critic_net(state_tensor)
                    BatchLogProbs  = action_dist.log_prob(sampled_action).sum(dim=-1)

                MemoryActions = []
                for i, con in enumerate(ConList):
                    steer_val = float(torch.clamp(sampled_action[i, 0], -1.0,  1.0).item())
                    accel_val = float(torch.clamp(sampled_action[i, 1],  0.0,  1.0).item())
                    brake_val = float(torch.clamp(sampled_action[i, 2],  0.0,  1.0).item())
                    MemoryActions.append((steer_val, accel_val, brake_val))

                    if BatchDones[i]:
                        con.send('TERMINATE')
                    else:
                        con.send((steer_val, accel_val, brake_val))

                NextStates      = []
                PreviousRewards = []
                PreviousDones   = []
                for con in ConList:
                    np_obs, reward, RaceDone = con.recv()
                    NextStates.append(np_obs)
                    PreviousRewards.append(reward)
                    PreviousDones.append(RaceDone)

                for i in range(NUM_WORKERS):
                    transition = {
                        'state':    BatchStates[i],
                        # Exactly 3 action dims — no padding zeros
                        'action':   np.array([MemoryActions[i][0],
                                              MemoryActions[i][1],
                                              MemoryActions[i][2]], dtype=np.float32),
                        'reward':   PreviousRewards[i],
                        'value':    state_value[i].item(),
                        'log_prob': BatchLogProbs[i].item(),
                        'done':     PreviousDones[i],
                    }
                    buffer.append(transition)

                total_episode_reward += sum(PreviousRewards)
                BatchStates = NextStates
                BatchDones  = PreviousDones

                # Early exit if all workers finished
                if all(BatchDones):
                    print(f'  All workers finished at step {step + 1}')
                    break

            # ── End of rollout ─────────────────────────────────────────────────
            mean_reward = total_episode_reward / NUM_WORKERS
            print(f'Episode {episode + 1}/{NUM_EPISODES} | '
                  f'Mean Reward: {mean_reward:.3f} | '
                  f'Buffer: {len(buffer)}')

            update_ppo(buffer)

            if mean_reward > best_reward:
                best_reward      = mean_reward
                patience_counter = 0
                torch.save(actor_net.state_dict(),  'best_actor.pth')
                torch.save(critic_net.state_dict(), 'best_critic.pth')
                print(f'  ✓ New best: {best_reward:.4f} — models saved.')
            else:
                patience_counter += 1
                print(f'  No improvement. Patience: {patience_counter}/{PATIENCE_LIMIT}')

            if patience_counter >= PATIENCE_LIMIT:
                print(f'Early stopping after {episode + 1} episodes.')
                break

            # Terminate worker processes cleanly
            for con in ConList:
                con.send('TERMINATE')
            for process in ProcessList:
                process.join(timeout=10)

    finally:
        # Safety cleanup
        for con in ConList:
            try:
                con.send('TERMINATE')
            except Exception:
                pass
        for process in ProcessList:
            process.join(timeout=5)


In [8]:
if __name__ == '__main__':
    main()


Episode 1/500 | Mean Reward: -40.357 | Buffer: 5000
  [PPO] actor_loss=1.7836  critic_loss=71081.8750  entropy=2.7583  approx_kl=-0.5886  actor_grad=2.8112
  ✓ New best: -40.3567 — models saved.
Episode 2/500 | Mean Reward: 15604.806 | Buffer: 5000
  [PPO] actor_loss=1.3122  critic_loss=642618.8125  entropy=2.7598  approx_kl=-0.4556  actor_grad=2.5078
  ✓ New best: 15604.8062 — models saved.
Episode 3/500 | Mean Reward: 12467.011 | Buffer: 5000
  [PPO] actor_loss=3.2137  critic_loss=302011.8438  entropy=2.7631  approx_kl=-0.6823  actor_grad=4.4004
  No improvement. Patience: 1/60
Episode 4/500 | Mean Reward: 1769.035 | Buffer: 5000
  [PPO] actor_loss=1.7923  critic_loss=192093.8594  entropy=2.7683  approx_kl=-0.8030  actor_grad=5.0238
  No improvement. Patience: 2/60
Episode 5/500 | Mean Reward: 8522.960 | Buffer: 5000
  [PPO] actor_loss=1.9486  critic_loss=289044.3125  entropy=2.7731  approx_kl=-0.7228  actor_grad=12.3941
  No improvement. Patience: 3/60
Episode 6/500 | Mean Reward: 9